# 4. Use LlamaIndex's prebuilt ReAct agent

The previous notebook makes retrieval, grading and rewriting explicit. Here LlamaIndex's `ReActAgent` manages the loop and decides when to search again. It uses the same GoodMem collections, reranker, prompts and native retrieval tools.

The code uses the current workflow-based API: construct the agent and `await agent.run(...)`. The older `ReActAgent.from_tools(...).chat(...)` example in the published GoodMem plugin no longer works with the tested LlamaIndex version.

In [1]:
from goodmem_rag.config import Settings, chat_model
from goodmem_rag.retrieval import make_tools

settings = Settings.from_env()
state = settings.state()
model = chat_model()

In [2]:
from llama_index.core.agent.workflow import ReActAgent
from goodmem_rag.agents import SYSTEM_PROMPT
from goodmem_rag.evaluation import SEQUENTIAL_QUESTION

async with settings.async_client() as client:
    tools = make_tools(async_client=client, state=state)
    agent = ReActAgent(llm=model, tools=tools, system_prompt=SYSTEM_PROMPT,
                       streaming=False, timeout=180, early_stopping_method="generate")
    response = await agent.run(user_msg=SEQUENTIAL_QUESTION, max_iterations=5)

print(response.response.content)
for call in response.tool_calls:
    print(call.tool_name, call.tool_kwargs)
    print("Sources:", [node.metadata.get("source") for node in call.tool_output.raw_output])

The recommended higher-level framework for prebuilt agent architectures is LangChain's agents (https://docs.langchain.com/oss/python/langchain/agents). 

In LangChain, the agent constructor is `create_agent`. The tool-calling loop works as follows:

1. The model's response includes a request to execute a tool.
2. The agent loop handles the tool execution, passing the results back to the model for subsequent reasoning.
3. This creates a conversation loop where the model can use tool results to generate its final response.

Here’s an example of creating an agent and invoking it:

```python
from langchain.agents import create_agent
from langchain.tools import fetch_order_status
from langchain.chat_models import ChatOpenAI

agent = create_agent(
    ChatOpenAI(model="ollama:north-mini-code-1.0"),
    tools=[fetch_order_status],
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "What is the status of order #12345?"}]
})
```

In this example, the agent handles the tool 

## Compare the two patterns

The explicit workflow gives you a place to enforce a relevance policy or add a review step. ReAct has less application code and lets the model decide how to proceed.

Both use server-side reranking without a GoodMem LLM. The chat model writes the final answer. Run `uv run llamaindex-rag evaluate` to check retrieval, direct answers, single-collection questions, comparisons and dependent searches for both agents. Those checks establish that the port works; comparing answer quality needs a larger controlled evaluation.